# CARA run 2 (v2): all 13 ETH3D scenes, extra VGGT signals, relative translations

This run replaces nothing from run 1; it writes to a new folder `out_eth3d_v2`.
New in run 2: every scene is used for testing (the analysis rotates which scenes calibrate), new random sub-scenes (seed 3),
and each file also stores VGGT's relative translation, the previous camera-head iteration, and per-image confidence statistics.

**Before you start:** Runtime > Change runtime type > **GPU**. Upload the new `CARA_GPU_kit_v2.zip` to `MyDrive/CARA/`.
The ETH3D archive from run 1 is reused (no new download). About 780 files, roughly 8-9 GPU hours on a T4:
it resumes after every disconnect. After a disconnect run cells 1-5 again, then cell 6.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive and working folders
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/CARA'          # persistent (outputs, zips)
LOCAL = '/content/local'                       # fast, lost on disconnect (unzipped images)
os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
FREE_TIER = True    # True: reduced workload that fits free T4 sessions; False: full protocol (Colab Pro / L4 / A100)

In [ ]:
# 3. Install VGGT and the CARA kit (about 2 minutes)
# VGGT's packaging pins numpy<2, which downgrades Colab's numpy and breaks other packages.
# The part of VGGT we use (model, image loading, pose decoding) works with Colab's numpy 2,
# so we install VGGT WITHOUT its pinned dependencies and add only what it needs.
%cd /content
!test -d vggt || git clone -q https://github.com/facebookresearch/vggt.git
!pip -q install --no-deps -e /content/vggt
!pip -q install einops safetensors huggingface_hub pycolmap==4.2.0 gtsam
!unzip -oq {WORK}/CARA_GPU_kit_v2.zip -d /content/cara_sfm
%cd /content/cara_sfm/gpu_kit
!python -c "import numpy, torch, vggt; from vggt.models.vggt import VGGT; print('numpy', numpy.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| VGGT import OK')"


In [ ]:
# 4. ETH3D data (5.5 GB, direct download, kept on Drive; extracted to fast local disk each session)
import os
z = f'{WORK}/multi_view_training_dslr_undistorted.7z'
if not os.path.exists(z):
    !wget -c -q --show-progress -O {z}.part https://www.eth3d.net/data/multi_view_training_dslr_undistorted.7z && mv {z}.part {z}
!apt-get -qq install -y p7zip-full > /dev/null
if not os.path.exists(f'{LOCAL}/eth3d/facade'):
    !mkdir -p {LOCAL}/eth3d && 7z x -y -o{LOCAL}/eth3d {z} > /dev/null
!ls {LOCAL}/eth3d
!find {LOCAL}/eth3d/facade -maxdepth 2 | head -8

In [ ]:
# 5. Settings (fixed in advance; do not change after seeing results)
ETH = f'{LOCAL}/eth3d'
OUT = f'{WORK}/out_eth3d_v2'
CACHE = f'{WORK}/out_eth3d/_degraded'     # degraded images from run 1 are reused
V2 = dict(n_sub=12, per_scene=12, n_test_sub=8, k=6, batch=8, seed=3,
          conds='clean,fog0.7,fog0.5,fog0.3,low0.3,low0.15,rain0.5')
# 13 scenes x (8 sub-scenes x 7 conditions + 4 clean-only) = 780 files.
# If a run fails with 'CUDA out of memory', set batch=4 and re-run (finished files are kept).

In [ ]:
# 6. Run 2 front end, all 13 scenes. Resumable: re-run this cell after a disconnect.
#    Progress lines appear only for newly computed files; the first line can take 5-15 min (model download).
!python frontend_vggt.py --dataset eth3d --root {ETH} --out {OUT} --cache {CACHE} --role cv --seed {V2['seed']} \
   --n-sub {V2['n_sub']} --per-scene {V2['per_scene']} --n-test-sub {V2['n_test_sub']} --k {V2['k']} \
   --batch {V2['batch']} --conds {V2['conds']}

In [ ]:
# 7. Progress count (expect 780 when finished) and sanity check
!ls {OUT} | grep -c '^cv__'
!python check_frontend.py --in {OUT}

In [ ]:
# 8. Package for Claude (about 15-25 MB): send MyDrive/CARA/CARA_frontend_v2.zip
!python check_frontend.py --in {OUT} > {WORK}/check_frontend_v2.txt; cat {WORK}/check_frontend_v2.txt
!nvidia-smi --query-gpu=name --format=csv > {WORK}/gpu_v2.txt
!cd {WORK} && rm -f CARA_frontend_v2.zip && zip -qr CARA_frontend_v2.zip out_eth3d_v2 check_frontend_v2.txt gpu_v2.txt
!ls -la {WORK}/CARA_frontend_v2.zip